In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.functions import lit, col, current_timestamp

In [0]:
movie_df = (
    spark.read
        .table("movie_silver.movies")
        .filter(f"file_date = '{v_file_date}'")
)

In [0]:
countries_df = (
    spark.read
        .table("movie_silver.countries")
)

In [0]:
movie_companies_df = (
    spark.read
        .table("movie_silver.movies_companies")
        .filter(f"file_date = '{v_file_date}'")
)

In [0]:
production_countries_df = (
    spark.read
        .table("movie_silver.production_countries")
        .filter(col("file_date") == v_file_date)
)

In [0]:
production_companies_df = (
    spark.read
        .table("movie_silver.production_companies")
        .filter(col("file_date") == v_file_date)
)

In [0]:
results_country_prod_company = (
    movie_df.filter(movie_df["year_release_date"] > 2010)
        .join(production_countries_df,
              movie_df["movie_id"] == production_countries_df["movie_id"], "inner")
        .join(countries_df,
              production_countries_df["country_id"] == countries_df["country_id"], "inner")
        .join(movie_companies_df,
              movie_df["movie_id"] == movie_companies_df["movie_id"], "inner")
        .join(production_companies_df,
              movie_companies_df["company_id"] == production_companies_df["company_id"], "inner")
).select(movie_df["title"], movie_df["budget"], movie_df["revenue"],
         movie_df["duration_time"],movie_df["release_date"],
         countries_df["country_name"],
         production_companies_df["company_name"]
).withColumn("created_date", lit(v_file_date))

In [0]:
display(results_country_prod_company)

In [0]:
overwrite_partitioned_table("movie_gold", "results_country_prod_company", "created_date", v_file_date)

In [0]:
(
    results_country_prod_company
    .write
        .mode("append")
        .partitionBy("created_date")
        .saveAsTable(f"movie_gold.results_country_prod_company")
)